## Ch9-03 -- Stale detection, at scale

This notebook introduces `check_stale()` applied across a small set of tracked records at once, not just one; after running it you can tell, for each of two real records, whether it is still current against the real, committed model, and watch what a real model edit actually does to each one.

Chapter 8 notebook 03 already showed `check_stale()` on one record against one lemma. This notebook keeps the chapter's original filename (`03-stale-detection.ipynb`): the content is genuinely about scale -- checking several tracked records against the real, current model in one pass -- but "at scale" over two records is still a small, honestly-scoped demonstration, not a claim that every record this tutorial has ever produced is being tracked (notebook 02's own scope note applies here too). It reuses notebook 02's own two records, `AS-C06` and `AS-C08`, rebuilt verbatim below (the exact same field strings notebook 02 uses) since each notebook in this tutorial loads and runs independently.

In [1]:
from pathlib import Path
import opensysml
from toaster.report import format_diagnostics

conn = opensysml.connect(version="v0.9.0")
source = Path("../../models/ch08-cumulative.sysml").read_text()
model = conn.load_from_content(source, strict=False)
assert model.ok, f"Model failed: {format_diagnostics(model.diagnostics)}"


A record with an empty identifier fails validation outright, distinct from staleness: it was never a valid record to begin with, regardless of which model it references (the same control Chapter 8 notebook 03 used).

In [2]:
from toaster.evidence import ReviewRecord, hash_content, validate_record

broken = ReviewRecord(
    identifier="",        # intentionally empty
    kind="asserted_solution",
    claim="Some claim",
    model_ref="ToasterDemo",
    content_hash=hash_content(source),
    scope="Some scope",
    criteria="Some criteria",
    rationale="Some rationale",
    counterevidence="Some counterevidence",
    record_kind="worked_example",
)
errors = validate_record(broken)
assert len(errors) > 0, "Expected validation errors for empty identifier"
print(f"Negative control ok: errors={errors}")


Negative control ok: errors=['identifier is empty']


`AS-C06` and `AS-C08`, rebuilt below **verbatim**, the exact same field strings notebook 02 uses (see notebook 02 for the narration behind each field): `AS-C06`'s `content_hash` against `models/ch06-cumulative.sysml`, the model it was actually written against; `AS-C08`'s against `models/ch08-cumulative.sysml`, the real, current model this notebook also loads above.

In [3]:
ch06_source = Path("../../models/ch06-cumulative.sysml").read_text()

as_c06 = ReviewRecord(
    identifier="AS-C06",
    kind="asserted_solution",
    claim=(
        "ResistanceCoil, an electrically switched resistive element that converts "
        "energy to heat by Joule heating, is selected over a combustion-based "
        "alternative (a gas burner, the tongs-and-blowtorch alternative this "
        "tutorial already contrasts) as the mechanism HeatGenerator commits to."
    ),
    model_ref="ToasterDemo::ResistanceCoil",
    content_hash=hash_content(ch06_source),
    scope="ToasterDemo::HeatGenerator and its realizations",
    criteria=(
        "The chosen mechanism must pair with the discrete timing control "
        "ControlSystem's durationOut already provides, and must expose a rating "
        "heatGenerationReq's power threshold can be checked against once a "
        "concrete part exists."
    ),
    premises=[
        "ControlSystem declares durationOut : DurationPort (Chapter 5), a "
        "discrete duration signal, confirmed by model.find() above.",
        "Domain premise, not derived from the model: a resistive element "
        "responds to being switched on and off directly, while a combustion "
        "source needs separate ignition and fuel-metering machinery to do "
        "the same. Neither HeatGenerator nor any of its realizations is yet "
        "connected to ControlSystem's port in this model; this premise is "
        "about the physical mechanisms themselves, not about what the model "
        "currently wires together.",
    ],
    assumption_refs=[
        "HeatGenerator::energyIn and GenerateHeat carry no energy-form commitment "
        "(notebook 01): this record is what actually commits to an electrical "
        "form, not a fact already built into the port or the function."
    ],
    evidence_refs=[
        "model.find('ToasterDemo::ControlSystem::durationOut') resolves to a "
        "real portUsage, confirmed above."
    ],
    rationale=(
        "An electrically resistive element responds to being switched on "
        "and off directly, the same shape as duration's discrete timing "
        "signal, while a combustion-based burner needs separate ignition "
        "and fuel-metering machinery to respond the same way (the domain "
        "premise above). That is a claim about how the two mechanisms "
        "work, not something this model currently shows: no realization "
        "of HeatGenerator is yet connected to ControlSystem's "
        "durationOut port, so this selection is a reasoned engineering "
        "preference argued from mechanism, not a claim that the model "
        "already connects one mechanism and not the other."
    ),
    counterevidence=(
        "This does not rule out a combustion design: a burner controlled by its "
        "own timed valve could equally use a duration-like signal, which is "
        "exactly why the argument above rests on a domain premise about how "
        "the two mechanisms work, not on anything the model itself already "
        "builds or connects. Joule heating's own relation (power proportional "
        "to resistance and the square of current) is still not modeled, so "
        "efficiency and response-time comparisons remain out of reach "
        "either way."
    ),
    residual_uncertainties=(
        "Once a supply and a control policy are modeled together, this "
        "selection could be revisited against a real trade study rather than "
        "a domain premise alone."
    ),
    disposition="pending",
    dependency_freshness="current",
    engineering_conclusion="undetermined",
    record_kind="worked_example",
)

errors = validate_record(as_c06)
print(f"AS-C06 validation errors: {errors}")


AS-C06 validation errors: []


`AS-C08`, the record Chapter 8's own Z3 proof grounds, rebuilt the same verbatim way.

In [4]:
as_c08 = ReviewRecord(
    identifier="AS-C08",
    kind="asserted_solution",
    claim=(
        "deliveredEnergyBoundedBySupply, a hand-restated real-arithmetic lemma of the same "
        "shape as HeatGenerator's efficiencyBounded constraint and deliveredEnergy's own "
        "definition, holds for every value of efficiency in [0,1] and every non-negative "
        "power and duration a companion restatement admits."
    ),
    model_ref="ToasterDemo::deliveredEnergyBoundedBySupply",
    content_hash=hash_content(source),
    scope=(
        "The lemma governs the companion restatement's own heatGenCheck.efficiency, "
        "heatGenCheck.power and heatGenCheckDuration features; it is not a solver-checked "
        "reference to HeatGenerator's own efficiencyBounded or deliveredEnergy (DEFERRED.md "
        "D-030, D-031), only a hand-restated copy of the same shape."
    ),
    criteria=(
        "verify_holds() reports a single satisfied verdict for deliveredEnergyBoundedBySupply, "
        "with the reason text naming z3 (not propagation alone), proved for all values of the "
        "unbound heatGenCheck.efficiency, heatGenCheck.power and heatGenCheckDuration features."
    ),
    premises=[],
    assumption_refs=[],
    evidence_refs=[
        "verify_holds: deliveredEnergyBoundedBySupply satisfied "
        "(z3: holds for all values of unbound features)"
    ],
    rationale=(
        "verify_holds() calls sysml-toolkit's real verify --solve command, which runs Z3 "
        "over the unbound features of a companion restatement of this lemma (see the "
        "narration above for why a companion file is used) and reports "
        "deliveredEnergyBoundedBySupply as satisfied: proved for all values the restatement "
        "admits, not read back from one entered value. This is a materially different kind of "
        "evidence from an evaluate-only verdict: verify_satisfaction() could only ever check "
        "a relation at whichever single power, duration and efficiency a candidate happens to "
        "carry. It is also, deliberately, a narrower claim than 'this proves HeatGenerator's "
        "own conservation property': see counterevidence."
    ),
    counterevidence=(
        "This proof is NOT a solver-checked reference to HeatGenerator's own "
        "efficiencyBounded constraint or deliveredEnergy calc: this toolchain's Z3 backend "
        "does not compose two separately declared assert constraints, whether sibling or "
        "inherited (DEFERRED.md D-030), and cannot reason through a chained calc invocation "
        "like heatGenCheck.deliveredEnergy(...) (D-031). Confirmed directly: loosening "
        "efficiencyBounded's own literal bound to <= 1.5, or doubling deliveredEnergy's own "
        "definition by a factor of 2.0, in the real committed model changes neither the "
        "original elements' own verdicts nor this lemma's verdict at all, because the lemma "
        "restates its own copy of both rather than referencing either. The companion file "
        "used by verify_holds also restates the construct rather than checking the "
        "committed model directly, because toaster.modelcheck's own text parser does not "
        "yet handle the extra annotation the CLI prints for assert satisfy declarations "
        "(DEFERRED.md D-029)."
    ),
    residual_uncertainties=(
        "Whether efficiency, power and duration ever take values outside the bound in a "
        "real candidate is not addressed by this proof; it establishes only that the "
        "restated lemma respects conservation wherever its own bound is honored. If "
        "HeatGenerator's own efficiencyBounded or deliveredEnergy is ever edited, this "
        "record's content_hash (computed from the whole model file) does go stale, which "
        "forces a re-review, but nothing automatically re-checks that the restated copy "
        "still matches the edited original; that check would be manual. No physical heat "
        "generator has been checked against this property; HeatGenerator remains an "
        "abstract carrier with no concrete realization of its own."
    ),
    disposition="pending",
    dependency_freshness="current",
    engineering_conclusion="supported",
    record_kind="worked_example",
)

errors = validate_record(as_c08)
print(f"AS-C08 validation errors: {errors}")


AS-C08 validation errors: []


Checked against `models/ch08-cumulative.sysml` before any edit: `AS-C06` was written against `models/ch06-cumulative.sysml`, and is already stale, but not from mere unrelated bookkeeping. `AS-C08` was written against this exact file, so it is still current.

In [5]:
from toaster.evidence import check_stale

tracked = {"AS-C06": as_c06, "AS-C08": as_c08}

print("Before any edit:")
for identifier, record in tracked.items():
    print(f"  {identifier}: stale={check_stale(record, source)}")

assert check_stale(as_c06, source) is True
assert check_stale(as_c08, source) is False


Before any edit:
  AS-C06: stale=True
  AS-C08: stale=False


`AS-C06`'s own `counterevidence` names something specific as still missing: "Joule heating's own relation ... is still not modeled, so efficiency and response-time comparisons remain out of reach either way." A direct diff between `models/ch06-cumulative.sysml` and the current model shows that gap partly overtaken, not exactly closed: Chapter 7 added `efficiency`, `efficiencyBounded` and `deliveredEnergy` to `HeatGenerator`, squarely inside the scope `AS-C06` itself declares (`ToasterDemo::HeatGenerator and its realizations`), so an efficiency comparison can now at least be started. The Joule-heating relation the same counterevidence names, and any response-time comparison, are still not modeled. `AS-C06` is not just formally stale, a hash mismatch; its own stated uncertainty has been substantively overtaken by real, subsequent model growth -- a record that could now, at least in part, actually be re-checked against something that did not previously exist.

In [6]:
import difflib

ch06_lines = ch06_source.splitlines()
ch08_lines = source.splitlines()
added = [
    line for line in difflib.unified_diff(ch06_lines, ch08_lines, lineterm="")
    if line.startswith("+") and not line.startswith("+++")
]
efficiency_lines = [line for line in added if "efficiency" in line.lower()]
print(f"{len(efficiency_lines)} added lines mention efficiency, including:")
for line in efficiency_lines[:3]:
    print(f"  {line.strip()}")

assert efficiency_lines, "Expected efficiency-related growth between ch06 and the current model"


18 added lines mention efficiency, including:
  +        attribute efficiency : DimensionOneValue;
  +        assert constraint efficiencyBounded {
  +            doc /* Efficiency is the fraction of supplied energy delivered as


`AS-C08`'s own `residual_uncertainties` names a different, specific risk: editing `HeatGenerator`'s own `efficiencyBounded` or `deliveredEnergy` would stale this record. The edit applied next is NOT that: it loosens `deliveredEnergyBoundedBySupply`'s own bound (the companion lemma copy the record is actually about), a different part of the same file. `check_stale()` still reports the record stale, for a reason its own text is explicit about: `content_hash` is computed from the WHOLE model file, so any edit anywhere in it, not only to the specific elements a record's own residual names, invalidates the hash -- a real, partial safeguard, honestly not a check that the specifically-named risk is what actually happened.

In [7]:
# Loosen the lemma's own bound (the companion copy AS-C08 is actually about), not
# HeatGenerator's efficiencyBounded/deliveredEnergy (the elements its own residual
# names as a risk) -- a different part of the same file.
revised_source = source.replace(
    "heatGenCheck.efficiency <= 1.0",
    "heatGenCheck.efficiency <= 1.2",
)
assert revised_source != source, "Expected the replacement to change the source"
assert "0.0 <= efficiency and efficiency <= 1.0" in revised_source, (
    "efficiencyBounded's own bare-`efficiency` bound must be untouched by this edit"
)
revised_model = conn.load_from_content(revised_source, strict=False)
assert revised_model.ok, "Revised model should still parse"

print("After loosening deliveredEnergyBoundedBySupply's own bound (1.0 -> 1.2):")
for identifier, record in tracked.items():
    print(f"  {identifier}: stale={check_stale(record, revised_source)}")

assert check_stale(as_c06, revised_source) is True
assert check_stale(as_c08, revised_source) is True
conn.close()


After loosening deliveredEnergyBoundedBySupply's own bound (1.0 -> 1.2):
  AS-C06: stale=True
  AS-C08: stale=True


Both records now report stale, but the two histories are genuinely different. `AS-C06` was stale before this edit too, and for a reason this specific edit has nothing to do with: real growth (Chapter 7) partly overtook the gap its own counterevidence named, without closing it (the Joule relation and response-time comparisons are still unmodeled). `AS-C08` newly went stale from an edit to a different construct than the one its own residual specifically warned about, caught only because the hash covers the whole file. Reading each record's own fields against the real model diff, not just its stale/current status, is what surfaces why each one changed: checking either record alone, at either point, would only have reported "stale" or "current," not the reason.

Two records, tracked and checked together against the same file before and after one real edit, reported two different histories: one substantively overtaken by real growth, the other caught by a coarse, whole-file safeguard rather than a targeted one.

Try the chapter exercise in `exercises/ch09/exercise.ipynb`: it asks you to check your own two records for staleness, at scale, the same way this notebook checks `AS-C06` and `AS-C08` together.